In [1]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split, cross_validate, GridSearchCV
import pickle

# Load ratings data
ratings = pd.read_csv('data/ml-latest-small/ratings.csv')
movies = pd.read_csv('data/ml-latest-small/movies.csv')

# Load data into Surprise format
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

print(f"Dataset loaded: {len(ratings)} ratings")
print(f"Users: {ratings['userId'].nunique()}")
print(f"Movies: {ratings['movieId'].nunique()}")

Dataset loaded: 100836 ratings
Users: 610
Movies: 9724


In [2]:
# Split data 80/20
trainset, testset = train_test_split(data, test_size=0.20, random_state=42)

# Train baseline SVD model with default parameters
model = SVD(random_state=42)
model.fit(trainset)

# Evaluate on test set
predictions = model.test(testset)
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

print(f"\nBaseline SVD Performance:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

RMSE: 0.8807
MAE:  0.6766

Baseline SVD Performance:
RMSE: 0.8807
MAE:  0.6766


In [3]:
# Grid search over key SVD hyperparameters
param_grid = {
    'n_factors': [50, 100, 150],    # Number of latent factors
    'n_epochs': [20, 30],            # Number of training iterations
    'lr_all': [0.005, 0.010],        # Learning rate
    'reg_all': [0.02, 0.10]          # Regularization
}

gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=5, n_jobs=-1)
gs.fit(data)

print("Best RMSE score:", round(gs.best_score['rmse'], 4))
print("Best MAE score:", round(gs.best_score['mae'], 4))
print("\nBest parameters:")
print(gs.best_params['rmse'])

Best RMSE score: 0.8546
Best MAE score: 0.6559

Best parameters:
{'n_factors': 150, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}


In [4]:
# Train final model with best parameters
best_params = gs.best_params['rmse']
final_model = SVD(
    n_factors=best_params['n_factors'],
    n_epochs=best_params['n_epochs'],
    lr_all=best_params['lr_all'],
    reg_all=best_params['reg_all'],
    random_state=42
)

# Train on full trainset, evaluate on testset
final_model.fit(trainset)
predictions = final_model.test(testset)

rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

print(f"Final Model Performance:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

# Save model to disk
with open('model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

print("\nModel saved to model.pkl")

RMSE: 0.8621
MAE:  0.6608
Final Model Performance:
RMSE: 0.8621
MAE:  0.6608

Model saved to model.pkl


In [5]:
from collections import defaultdict

def precision_recall_at_k(predictions, k=10, threshold=3.5):
    """Calculate precision and recall at k for all users"""
    
    # Group predictions by user
    user_predictions = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_predictions[uid].append((est, true_r))
    
    precisions = []
    recalls = []
    
    for uid, user_ratings in user_predictions.items():
        # Sort by estimated rating, highest first
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        
        # Top k recommendations
        top_k = user_ratings[:k]
        
        # Number of relevant items in top k (estimated >= threshold)
        n_relevant_and_recommended = sum(1 for est, true_r in top_k if est >= threshold)
        
        # Number of relevant items overall (true rating >= threshold)
        n_relevant = sum(1 for est, true_r in user_ratings if true_r >= threshold)
        
        precision = n_relevant_and_recommended / k
        recall = n_relevant_and_recommended / n_relevant if n_relevant > 0 else 0
        
        precisions.append(precision)
        recalls.append(recall)
    
    avg_precision = sum(precisions) / len(precisions)
    avg_recall = sum(recalls) / len(recalls)
    
    return avg_precision, avg_recall

precision, recall = precision_recall_at_k(predictions, k=10, threshold=3.5)

print("Precision@10:", round(precision, 4))
print("Recall@10:   ", round(recall, 4))

Precision@10: 0.6916
Recall@10:    0.6927


In [6]:
def get_recommendations(user_id, model, ratings_df, movies_df, n=10):
    # Get list of all movie IDs
    all_movie_ids = ratings_df['movieId'].unique()
    
    # Get movies the user has already rated
    rated_movie_ids = ratings_df[ratings_df['userId'] == user_id]['movieId'].unique()
    
    # Filter to only unrated movies
    unrated_movies = [mid for mid in all_movie_ids if mid not in rated_movie_ids]
    
    # Predict ratings for all unrated movies
    predictions = [model.predict(user_id, mid) for mid in unrated_movies]
    
    # Sort by estimated rating descending
    predictions.sort(key=lambda x: x.est, reverse=True)
    
    # Get top n movie IDs and scores
    top_n = predictions[:n]
    
    # Build results dataframe
    results = pd.DataFrame([{
        'movieId': pred.iid,
        'estimated_rating': round(pred.est, 2)
    } for pred in top_n])
    
    results = results.merge(movies_df[['movieId', 'title', 'genres']], on='movieId')
    return results[['title', 'genres', 'estimated_rating']]

# Test with user 1
recommendations = get_recommendations(1, final_model, ratings, movies, n=10)
print("Top 10 recommendations for User 1:")
print(recommendations.to_string(index=False))

Top 10 recommendations for User 1:
                                                 title                     genres  estimated_rating
Wallace & Gromit: The Best of Aardman Animation (1996) Adventure|Animation|Comedy              5.00
      Three Billboards Outside Ebbing, Missouri (2017)                Crime|Drama              5.00
                   Guess Who's Coming to Dinner (1967)                      Drama              5.00
                               Band of Brothers (2001)           Action|Drama|War              5.00
                                 Secrets & Lies (1996)                      Drama              5.00
                                 Paths of Glory (1957)                  Drama|War              5.00
                                     Adam's Rib (1949)             Comedy|Romance              5.00
                      Shawshank Redemption, The (1994)                Crime|Drama              5.00
                                      Submarine (2010)       Come